# 课后练习解答（02.06_training_practice）

本解答对应章节课后练习，共 15 题。

### 问题1（单选题）

**题目：** 保存 checkpoint 时，保存 state_dict 而不是整个模型的主要原因是？
A. 文件更小、结构解耦、便于跨版本加载
B. 训练更快
C. 精度更高
D. 无需保存权重

**解答：** A

**解析：** state_dict 只保存参数映射，加载时由代码重新创建结构，避免 pickle 版本和类定义耦合问题。


### 问题2（单选题）

**题目：** 最佳 checkpoint 应依据哪个指标选择？
A. 验证集 Top-1
B. 最后一个 epoch 的训练 loss
C. 训练集准确率
D. 随机选择

**解答：** A

**解析：** 只有验证集指标能代表未见数据的泛化能力。


### 问题3（多选题）

**题目：** 断点续训需要保存？
A. model.state_dict()
B. optimizer.state_dict()
C. scheduler.state_dict()
D. epoch/step 与随机种子

**解答：** ABCD

**解析：** 恢复训练必须同时恢复模型、优化器、调度器与进度信息，否则无法严格续训。


### 问题4（多选题）

**题目：** DataLoader num_workers 设置过大可能带来？
A. 共享内存/内存压力增大
B. worker 启动开销增大
C. 数据加载不一定更快
D. 训练精度下降

**解答：** ABC

**解析：** worker 数量超过系统资源后会引发调度和内存瓶颈，但不影响模型精度。


### 问题5（判断题）

**题目：** 每个训练 step 应依次执行 optimizer.zero_grad()、loss.backward()、optimizer.step()。

**解答：** 对

**解析：** 先清空旧梯度，再反向传播，最后更新参数，顺序错误会导致梯度累积。


### 问题6（判断题）

**题目：** 把 model.state_dict() 加载到结构完全不同的模型上一定不会报错。

**解答：** 错

**解析：** load_state_dict 默认 strict=True，key 或 shape 不匹配会直接报错。


### 问题7（填空题）

**题目：** 防止上一个 batch 梯度累积到当前 step，应在 backward 前调用 ____。

**解答：** optimizer.zero_grad()


### 问题8（填空题）

**题目：** 验证阶段使用 ____ 关闭梯度计算，避免保存反向图。

**解答：** torch.no_grad()


### 问题9（简答题）

**题目：** 为什么验证评估放在每个 epoch 或固定间隔而不是每个 batch？

**解答：** 验证需要遍历完整验证集才有统计意义；每个 batch 评估会引入大量调度开销，且单 batch 结果波动大，无法反映整体泛化。


### 问题10（简答题）

**题目：** 提前停止应同时观察训练 loss 与验证指标，为什么不能只观察训练 loss？

**解答：** 训练 loss 只反映拟合训练集，过拟合时训练 loss 仍下降；验证指标才反映泛化，因此要以验证指标不再提升作为停止依据。


### 问题11（代码设计题）

**题目：** 编写 resume_training 函数：从 checkpoint 恢复 epoch、model、optimizer、scheduler 并继续训练。

**解答：** ```python
def resume_training(checkpoint_path, model, optimizer, scheduler):
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt.get("epoch", 0)
    seed = ckpt.get("seed")
    return start_epoch, seed
```


### 问题12（单选题）

**题目：** 训练 loss 下降但验证 loss 连续 5 轮上升，首选操作是？
A. 回滚到验证最佳 checkpoint 并停止或降低学习率
B. 增大学习率
C. 删除数据增强
D. 增加训练 epoch

**解答：** A

**解析：** 过拟合信号明确，应保留泛化最好的权重并调整训练强度。


### 问题13（多选题）

**题目：** 训练时显存 OOM 的可能原因包括？
A. batch_size 过大
B. num_workers 过多导致内存峰值高
C. 保存了过多中间激活
D. 使用 bf16

**解答：** ABC

**解析：** bf16 反而降低显存；前向激活、数据缓冲和批量大小才是 OOM 主因。


### 问题14（判断题）

**题目：** torch.no_grad() 下验证会显著降低显存占用，因为不构建反向图。

**解答：** 对

**解析：** 不保存中间激活的梯度信息，推理图可被释放。


### 问题15（简答题）

**题目：** 要实现严格可复现训练，需要控制哪些因素？请至少列出 4 项。

**解答：** 1) 固定全局随机种子并分别设置 torch/numpy/random；2) 固定 DataLoader shuffle 与 worker 种子；3) 固定模型初始化与权重；4) 固定 CUDA/NPU 算子确定性设置；5) 固定环境版本（torch、torch_npu、CANN、Python）。
